# Skeleton BERT self-supervised pretraining on Colab

This notebook runs the How2Sign keypoint pipeline on a Colab GPU. Before running, choose **Runtime → Change runtime type → GPU**. English translations and glosses are not used during pretraining.

For reliable I/O, keep archives and checkpoints in Google Drive, but extract the many small OpenPose/NPZ files into `/content` for training.

In [ ]:
import subprocess, sys
import torch
assert torch.cuda.is_available(), 'Enable a GPU runtime before continuing.'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)

## 1. Mount Drive and load the repository

Use a Git URL after this local repository has been pushed to GitHub, or upload the complete project folder to `MyDrive/sign_language_llm`. The project is copied to the Colab VM so imports and small-file access are fast.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil, subprocess

REPO_URL = ''  # e.g. https://github.com/USER/sign-language-llm.git
DRIVE_PROJECT = Path('/content/drive/MyDrive/sign_language_llm')
PROJECT = Path('/content/sign_language_llm')

if PROJECT.exists():
    shutil.rmtree(PROJECT)
if REPO_URL:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT)], check=True)
elif DRIVE_PROJECT.exists():
    shutil.copytree(DRIVE_PROJECT, PROJECT, ignore=shutil.ignore_patterns('.git', '.venv', 'data', 'outputs'))
else:
    raise FileNotFoundError('Set REPO_URL or upload the project to MyDrive/sign_language_llm')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT)], check=True)
print('Project ready:', PROJECT)

## 2. Stage How2Sign data locally

Put the official keypoint split archives in `MyDrive/sign_language_llm_data/archives`. Set the three filenames below to the actual downloaded archive names. Extraction can be skipped on reconnection if `/content/how2sign/raw` still exists.

In [ ]:
from pathlib import Path
import subprocess

DRIVE_ROOT = Path('/content/drive/MyDrive/sign_language_llm_data')
ARCHIVE_ROOT = DRIVE_ROOT / 'archives'
LOCAL_ROOT = Path('/content/how2sign')
RAW_ROOT = LOCAL_ROOT / 'raw'
PROCESSED_ROOT = LOCAL_ROOT / 'processed'
CHECKPOINT_ROOT = DRIVE_ROOT / 'checkpoints'
for path in (RAW_ROOT, PROCESSED_ROOT, CHECKPOINT_ROOT):
    path.mkdir(parents=True, exist_ok=True)

ARCHIVES = {
    'train': ARCHIVE_ROOT / 'TRAIN_ARCHIVE_NAME.tar.gz',
    'val': ARCHIVE_ROOT / 'VAL_ARCHIVE_NAME.tar.gz',
    'test': ARCHIVE_ROOT / 'TEST_ARCHIVE_NAME.tar.gz',
}
RUN_EXTRACTION = False  # set True after correcting archive names
if RUN_EXTRACTION:
    for split, archive in ARCHIVES.items():
        destination = RAW_ROOT / split
        destination.mkdir(parents=True, exist_ok=True)
        subprocess.run(['tar', '-xf', str(archive), '-C', str(destination)], check=True)

## 3. Convert OpenPose JSON to normalized skeleton sentences

Run conversion once per fresh Colab VM. For an initial pilot, point `RAW_ROOT` at a small subset or stop conversion after preparing a few thousand sentence folders.

In [ ]:
import subprocess

RUN_PREPARATION = False
if RUN_PREPARATION:
    for split in ('train', 'val', 'test'):
        subprocess.run([
            'sign-prepare',
            '--input-root', str(RAW_ROOT / split),
            '--output-root', str(PROCESSED_ROOT / 'clips' / split),
            '--manifest', str(PROCESSED_ROOT / f'{split}.jsonl'),
    ], check=True)

## 4. Create a Colab configuration

Checkpoints are written directly to Drive once per epoch. Training clips stay on the local VM. Start with 10 epochs for a pilot, then increase after confirming loss decreases.

In [ ]:
import json

with (PROJECT / 'configs/pretrain.json').open() as handle:
    config = json.load(handle)
config['data'].update({
    'train_manifest': str(PROCESSED_ROOT / 'train.jsonl'),
    'val_manifest': str(PROCESSED_ROOT / 'val.jsonl'),
    'num_workers': 2,
})
config['training'].update({
    'output_dir': str(CHECKPOINT_ROOT),
    'batch_size': 16,
    'epochs': 10,
    'amp': True,
})
COLAB_CONFIG = Path('/content/colab_pretrain.json')
COLAB_CONFIG.write_text(json.dumps(config, indent=2))
print(COLAB_CONFIG.read_text())

## 5. Train or resume

In [ ]:
import subprocess

RESUME = CHECKPOINT_ROOT / 'last.pt'
command = ['sign-pretrain', '--config', str(COLAB_CONFIG)]
if RESUME.exists():
    command += ['--resume', str(RESUME)]
print('Running:', ' '.join(command))
subprocess.run(command, check=True)

## 6. Export contextual word representations

Upload a test-only JSONL boundary file to Drive. It is used only here, after self-supervised training.

In [ ]:
BOUNDARIES = DRIVE_ROOT / 'annotations' / 'test_word_boundaries.jsonl'
REPRESENTATIONS = DRIVE_ROOT / 'representations' / 'sign_words.npz'
REPRESENTATIONS.parent.mkdir(parents=True, exist_ok=True)
subprocess.run([
    'sign-extract-words',
    '--checkpoint', str(CHECKPOINT_ROOT / 'best.pt'),
    '--manifest', str(PROCESSED_ROOT / 'test.jsonl'),
    '--boundaries', str(BOUNDARIES),
    '--output', str(REPRESENTATIONS),
], check=True)
print('Saved:', REPRESENTATIONS)

## 7. Create mBERT word references and run RSA

Upload `concepts_en.csv` with `id,text` columns. IDs must match the gloss IDs in the boundary file.

In [ ]:
CONCEPTS = DRIVE_ROOT / 'annotations' / 'concepts_en.csv'
TEXT_REPRESENTATIONS = DRIVE_ROOT / 'representations' / 'mbert_en_words.npz'
RSA_OUTPUT = DRIVE_ROOT / 'rsa' / 'sign_vs_mbert_en.csv'
RSA_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
subprocess.run([
    'sign-embed-text', '--concepts', str(CONCEPTS),
    '--output', str(TEXT_REPRESENTATIONS),
], check=True)
subprocess.run([
    'sign-rsa', '--sign', str(REPRESENTATIONS),
    '--text', str(TEXT_REPRESENTATIONS),
    '--output', str(RSA_OUTPUT), '--permutations', '1000',
], check=True)
print('Saved:', RSA_OUTPUT)